In [1]:

import os
import json
from pathlib import Path
from typing import List, Tuple, Dict

import cv2
import h5py
import numpy as np
import yaml
from ultralytics import YOLO

ROOT = Path.cwd()
TRAIN_DIR = ROOT / 'train'
TEST_DIR = ROOT / 'test'
IMAGES_LAB5_DIR = ROOT / 'images_lab5'

YOLO_CFG_DIR = ROOT / '.yolo_cfg'
YOLO_CFG_DIR.mkdir(parents=True, exist_ok=True)
os.environ['YOLO_CONFIG_DIR'] = str(YOLO_CFG_DIR.resolve())

print('ROOT:', ROOT)
print('Train exists:', TRAIN_DIR.exists())
print('Test exists:', TEST_DIR.exists())
print('images_lab5 exists:', IMAGES_LAB5_DIR.exists())


ROOT: C:\Users\wiad_\PycharmProjects\ItmoCv
Train exists: True
Test exists: True
images_lab5 exists: True


In [2]:

def _decode_filename(h5_file: h5py.File, name_ref) -> str:
    data = h5_file[name_ref][()]
    data = np.array(data).reshape(-1)
    return ''.join(chr(int(x)) for x in data)


def _read_attr_values(h5_file: h5py.File, bbox_group: h5py.Group, attr_name: str) -> List[float]:
    attr = bbox_group[attr_name]
    values = attr[()]
    values = np.array(values)

    if values.dtype == np.object_ or values.dtype.kind == 'O':
        out = []
        for ref in values.reshape(-1):
            ref_data = h5_file[ref][()]
            out.append(float(np.array(ref_data).reshape(-1)[0]))
        return out

    flat = values.reshape(-1)
    return [float(v) for v in flat]


def svhn_to_yolo_labels(split_dir: Path, cleanup_old_txt: bool = False) -> Dict[str, int]:
    mat_path = split_dir / 'digitStruct.mat'
    if not mat_path.exists():
        raise FileNotFoundError(f'?? ?????? {mat_path}')

    if cleanup_old_txt:
        for txt_path in split_dir.glob('*.txt'):
            txt_path.unlink(missing_ok=True)

    total = 0
    written = 0

    with h5py.File(mat_path, 'r') as f:
        ds = f['digitStruct']
        names = ds['name']
        bboxes = ds['bbox']

        for i in range(len(names)):
            total += 1
            filename = _decode_filename(f, names[i][0])
            image_path = split_dir / filename
            if not image_path.exists():
                continue

            bbox_group = f[bboxes[i][0]]
            left = _read_attr_values(f, bbox_group, 'left')
            top = _read_attr_values(f, bbox_group, 'top')
            width = _read_attr_values(f, bbox_group, 'width')
            height = _read_attr_values(f, bbox_group, 'height')

            x1 = min(left)
            y1 = min(top)
            x2 = max(l + w for l, w in zip(left, width))
            y2 = max(t + h for t, h in zip(top, height))

            img = cv2.imread(str(image_path))
            if img is None:
                continue

            h_img, w_img = img.shape[:2]

            x1 = max(0.0, min(x1, w_img - 1.0))
            y1 = max(0.0, min(y1, h_img - 1.0))
            x2 = max(x1 + 1.0, min(x2, float(w_img)))
            y2 = max(y1 + 1.0, min(y2, float(h_img)))

            cx = ((x1 + x2) / 2.0) / w_img
            cy = ((y1 + y2) / 2.0) / h_img
            bw = (x2 - x1) / w_img
            bh = (y2 - y1) / h_img

            label_path = image_path.with_suffix('.txt')
            with open(label_path, 'w', encoding='utf-8') as lf:
                lf.write(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')
            written += 1

    return {'split': split_dir.name, 'indexed': total, 'labels_written': written}


def create_data_yaml(root: Path) -> Path:
    data = {
        'path': str(root.resolve()).replace('\\', '/'),
        'train': 'train',
        'val': 'test',
        'test': 'test',
        'names': {0: 'number'}
    }
    yaml_path = root / 'svhn_number.yaml'
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)
    return yaml_path


In [ ]:

stats_train = svhn_to_yolo_labels(TRAIN_DIR, cleanup_old_txt=True)
stats_test = svhn_to_yolo_labels(TEST_DIR, cleanup_old_txt=True)

data_yaml_path = create_data_yaml(ROOT)

stats = {
    'train': stats_train,
    'test': stats_test,
    'data_yaml': str(data_yaml_path)
}

with open(ROOT / 'lab5_dataset_stats.json', 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

stats


In [ ]:

EPOCHS = 8
IMGSZ = 416
BATCH = 64
DEVICE = 0

model = YOLO('yolov8n.pt')

train_result = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=4,
    project=str(ROOT / 'runs' / 'lab5'),
    name='svhn_yolov8n',
    exist_ok=True,
    patience=4,
    optimizer='AdamW',
    lr0=1e-3,
    cos_lr=True,
)

train_result


In [ ]:

best_weights = ROOT / 'runs' / 'lab5' / 'svhn_yolov8n' / 'weights' / 'best.pt'
model_best = YOLO(str(best_weights))

val_metrics = model_best.val(data=str(data_yaml_path), split='val', verbose=False)

metrics_dict = {
    'precision': float(val_metrics.box.mp),
    'recall': float(val_metrics.box.mr),
    'mAP@0.5': float(val_metrics.box.map50),
    'mAP@0.5:0.95': float(val_metrics.box.map)
}

with open(ROOT / 'lab5_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, ensure_ascii=False, indent=2)

metrics_dict


In [ ]:

def yolo_xywhn_to_xyxy(line: str, w: int, h: int) -> Tuple[float, float, float, float]:
    _, cx, cy, bw, bh = [float(x) for x in line.strip().split()]
    x1 = (cx - bw / 2.0) * w
    y1 = (cy - bh / 2.0) * h
    x2 = (cx + bw / 2.0) * w
    y2 = (cy + bh / 2.0) * h
    return x1, y1, x2, y2


def iou_xyxy(a: Tuple[float, float, float, float], b: Tuple[float, float, float, float]) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0


def compute_mean_iou(model: YOLO, test_dir: Path, conf: float = 0.25, max_images: int = 3000):
    image_paths = sorted(test_dir.glob('*.png'))
    if max_images is not None:
        image_paths = image_paths[:max_images]

    ious = []
    matched = 0

    for img_path in image_paths:
        lbl_path = img_path.with_suffix('.txt')
        if not lbl_path.exists():
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        with open(lbl_path, 'r', encoding='utf-8') as f:
            line = f.readline().strip()
        if not line:
            continue

        gt = yolo_xywhn_to_xyxy(line, w, h)

        pred = model.predict(source=str(img_path), conf=conf, iou=0.5, max_det=1, verbose=False)
        if len(pred) == 0 or pred[0].boxes is None or len(pred[0].boxes) == 0:
            continue

        pb = pred[0].boxes.xyxy[0].cpu().numpy().tolist()
        pr = (float(pb[0]), float(pb[1]), float(pb[2]), float(pb[3]))

        ious.append(iou_xyxy(gt, pr))
        matched += 1

    return {
        'mean_iou': float(np.mean(ious)) if ious else 0.0,
        'matched_predictions': matched,
        'evaluated_images': len(image_paths)
    }


iou_stats = compute_mean_iou(model_best, TEST_DIR, conf=0.25, max_images=3000)

with open(ROOT / 'lab5_iou.json', 'w', encoding='utf-8') as f:
    json.dump(iou_stats, f, ensure_ascii=False, indent=2)

iou_stats


In [ ]:

preds = model_best.predict(
    source=str(IMAGES_LAB5_DIR),
    conf=0.25,
    iou=0.5,
    save=True,
    project=str(ROOT / 'runs' / 'lab5'),
    name='street_infer',
    exist_ok=True,
)

print(ROOT / 'runs' / 'lab5' / 'street_infer')


In [ ]:

final_report = {
    'IoU_mean': iou_stats['mean_iou'],
    'Precision': metrics_dict['precision'],
    'Recall': metrics_dict['recall'],
    'mAP@0.5': metrics_dict['mAP@0.5'],
    'mAP@0.5:0.95': metrics_dict['mAP@0.5:0.95'],
}
final_report
